# Mapping assignments to network exposure

## A Fixed Public Topology

Let $R$ be the correlation matrix of the declared public industry returns. For each industry, retain its $k=2$ largest off-diagonal correlations and then make the edge set undirected. The resulting adjacency matrix is

$$
A_{ij}=\mathbf 1\{i\text{ selects }j\text{ or }j\text{ selects }i\},\qquad A_{ii}=0.
$$

Here, $A_{ij}=1$ records a descriptive link, $i$ and $j$ label industries, and $k$ controls the number of initially selected neighbors. This rule fixes a topology. It does not show that one industry's return causes another industry's return.

In [1]:
from pathlib import Path
import hashlib
import io
import itertools
import math
import sys
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT

import RiskLabAI
from RiskLabAI.causal_factor_analysis import exposure_mean_contrast

assert RiskLabAI.__version__ == "3.2.0"

FIGURES = BOOK_ROOT / "outputs" / "figures" / "07_network_interference"
TABLES = BOOK_ROOT / "outputs" / "tables" / "07_network_interference"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "07_network_interference"
DATA_FILE = ensure_dataset(BOOK_ROOT, "french_49_industries")
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    latex = latex.replace("\\begin{table}[htbp]", "\\begin{table}[htbp]\n\\centering", 1)
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def read_public_returns():
    with zipfile.ZipFile(DATA_FILE) as archive:
        lines = archive.read(archive.namelist()[0]).decode("latin1").splitlines()
    header = next(index for index, line in enumerate(lines) if line.startswith(",Agric"))
    ending = next(index for index in range(header + 1, len(lines)) if not lines[index].strip())
    raw = pd.read_csv(io.StringIO("\n".join(lines[header:ending])))
    raw = raw.rename(columns={raw.columns[0]: "Date"})
    raw["Date"] = pd.to_datetime(raw["Date"].astype(str), format="%Y%m%d")
    raw = raw.set_index("Date").replace([-99.99, -999.0], np.nan) / 100
    return raw.loc["2019-01-01":"2023-12-31"].dropna().copy()


def correlation_graph(correlation, neighbors):
    values = correlation.to_numpy()
    size = len(correlation)
    adjacency = np.zeros((size, size), dtype=int)
    for row in range(size):
        order = np.argsort(values[row])
        selected = [column for column in order[::-1] if column != row][:neighbors]
        adjacency[row, selected] = 1
    return np.maximum(adjacency, adjacency.T)


def line_graph(size):
    adjacency = np.zeros((size, size), dtype=int)
    for index in range(size - 1):
        adjacency[index, index + 1] = adjacency[index + 1, index] = 1
    return adjacency


def ring_graph(size, radius=1):
    adjacency = np.zeros((size, size), dtype=int)
    for index in range(size):
        for step in range(1, radius + 1):
            adjacency[index, (index + step) % size] = 1
            adjacency[index, (index - step) % size] = 1
    return adjacency


def star_graph(size):
    adjacency = np.zeros((size, size), dtype=int)
    adjacency[0, 1:] = 1
    adjacency[1:, 0] = 1
    return adjacency


def complete_graph(size):
    return np.ones((size, size), dtype=int) - np.eye(size, dtype=int)


def exposure_labels(adjacency, assignment, threshold=1):
    return ((adjacency @ np.asarray(assignment)) >= threshold).astype(int)


def exposure_probabilities(adjacency, assignment_probability, threshold=1):
    degrees = adjacency.sum(axis=1).astype(int)
    positive = np.array([
        1 - sum(math.comb(int(degree), count) * assignment_probability ** count * (1 - assignment_probability) ** (degree - count) for count in range(min(threshold, int(degree) + 1)))
        for degree in degrees
    ])
    return np.column_stack([1 - positive, positive])


def assignment_probability(assignment, probability):
    treated = int(np.asarray(assignment).sum())
    return probability ** treated * (1 - probability) ** (len(assignment) - treated)


def circle_positions(size):
    angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, size, endpoint=False)
    return np.column_stack([np.cos(angles), np.sin(angles)])


def draw_network(axis, adjacency, labels, node_values=None, title=""):
    positions = circle_positions(len(labels))
    for first in range(len(labels)):
        for second in range(first + 1, len(labels)):
            if adjacency[first, second]:
                axis.plot(positions[[first, second], 0], positions[[first, second], 1], color=PALETTE["grid"], linewidth=1.5, alpha=0.8, zorder=1)
    values = adjacency.sum(axis=1) if node_values is None else np.asarray(node_values)
    colors = [PALETTE["cyan"] if value <= np.median(values) else PALETTE["amber"] for value in values]
    axis.scatter(positions[:, 0], positions[:, 1], s=420, c=colors, edgecolors=PALETTE["text"], linewidths=1.0, zorder=2)
    for index, label in enumerate(labels):
        axis.text(positions[index, 0], positions[index, 1], str(label), ha="center", va="center", color=PALETTE["background"], fontsize=8.5, fontweight="bold", zorder=3)
    axis.set_title(title)
    axis.set_aspect("equal")
    axis.axis("off")
    return positions


def capture_failure(name, calculation):
    try:
        calculation()
        return {"Check": name, "Rejected": False, "Explanation": "Calculation continued"}
    except ValueError:
        return {"Check": name, "Rejected": True, "Explanation": "Stopped as required"}

public_returns = read_public_returns()
selected_returns = public_returns.iloc[:, :12]
correlation = selected_returns.corr()
adjacency = correlation_graph(correlation, 2)
industries = selected_returns.columns.to_list()
degrees = adjacency.sum(axis=1)
edges = []
for first in range(len(industries)):
    for second in range(first + 1, len(industries)):
        if adjacency[first, second]:
            edges.append({"First industry": industries[first], "Second industry": industries[second], "Correlation": correlation.iloc[first, second]})
edge_frame = pd.DataFrame(edges)

provenance = pd.DataFrame([
    {"Item": "Public source", "Value": "Kenneth R. French 49 Industry Portfolios"},
    {"Item": "Stored archive", "Value": DATA_FILE.name},
    {"Item": "Archive bytes", "Value": DATA_FILE.stat().st_size},
    {"Item": "SHA256", "Value": file_sha256(DATA_FILE)},
    {"Item": "Analysis start", "Value": str(selected_returns.index.min().date())},
    {"Item": "Analysis end", "Value": str(selected_returns.index.max().date())},
    {"Item": "Complete dates", "Value": len(selected_returns)},
    {"Item": "Selected industries", "Value": len(industries)},
    {"Item": "Download during execution", "Value": "None"},
])
assert file_sha256(DATA_FILE) == "8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71A7C1CD20BAB83040DE"
assert len(selected_returns) == 1258 and len(industries) == 12
save_table(provenance, "mapping_public_provenance", "Provenance of the public returns used only to fix a topology.", "tab:cim7-mapping-public-provenance", fit=True)

industry_summary = pd.DataFrame({
    "Industry": industries,
    "Mean daily return": selected_returns.mean().to_numpy(),
    "Daily volatility": selected_returns.std().to_numpy(),
    "Degree": degrees,
})
save_table(industry_summary, "mapping_public_industries", "Selected public industries and descriptive topology degrees.", "tab:cim7-mapping-public-industries", fit=True)
save_table(edge_frame, "mapping_public_edges", "Edges in the fixed public correlation topology.", "tab:cim7-mapping-public-edges", fit=True)

graph_summary = pd.DataFrame([
    {"Quantity": "Industries", "Value": len(industries)},
    {"Quantity": "Undirected edges", "Value": int(adjacency.sum() / 2)},
    {"Quantity": "Density", "Value": adjacency.sum() / (len(industries) * (len(industries) - 1))},
    {"Quantity": "Minimum degree", "Value": int(degrees.min())},
    {"Quantity": "Mean degree", "Value": degrees.mean()},
    {"Quantity": "Maximum degree", "Value": int(degrees.max())},
])
save_table(graph_summary, "mapping_public_graph_summary", "Summary of the fixed public correlation topology.", "tab:cim7-mapping-public-graph-summary")

scope = pd.DataFrame([
    {"Object": "Public returns", "Role": "Fix correlations and the descriptive edge set", "Causal conclusion": "None"},
    {"Object": "Graph edge", "Role": "Defines which assignments enter a supplied exposure rule", "Causal conclusion": "None by itself"},
    {"Object": "Generated assignment", "Role": "Supplies known randomization probabilities", "Causal conclusion": "Design based within the generated experiment"},
    {"Object": "Generated potential outcomes", "Role": "Supply known exposure responses", "Causal conclusion": "Local to the declared design"},
])
save_table(scope, "mapping_scope_ledger", "Roles and limits of the public and generated inputs.", "tab:cim7-mapping-scope-ledger", fit=True)

correlation_data = correlation.copy()
correlation_data.insert(0, "Industry", correlation_data.index)
correlation_data.to_csv(FIGURE_DATA / "mapping_public_correlation.csv", index=False)
figure, axis = plt.subplots(figsize=(8.4, 7.0), constrained_layout=True)
image = axis.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
axis.set_xticks(np.arange(len(industries)), industries, rotation=55, ha="right")
axis.set_yticks(np.arange(len(industries)), industries)
axis.set_title("Public Industry Return Correlations, 2019 to 2023")
figure.colorbar(image, ax=axis, label="Correlation", shrink=0.82)
save_figure(figure, "mapping_public_correlation")

positions = circle_positions(len(industries))
node_data = pd.DataFrame({"Industry": industries, "X": positions[:, 0], "Y": positions[:, 1], "Degree": degrees})
node_data.to_csv(FIGURE_DATA / "mapping_public_topology_nodes.csv", index=False)
edge_plot = edge_frame.copy()
edge_plot["First X"] = edge_plot["First industry"].map(dict(zip(industries, positions[:, 0])))
edge_plot["First Y"] = edge_plot["First industry"].map(dict(zip(industries, positions[:, 1])))
edge_plot["Second X"] = edge_plot["Second industry"].map(dict(zip(industries, positions[:, 0])))
edge_plot["Second Y"] = edge_plot["Second industry"].map(dict(zip(industries, positions[:, 1])))
edge_plot.to_csv(FIGURE_DATA / "mapping_public_topology_edges.csv", index=False)
figure, axis = plt.subplots(figsize=(8.4, 7.0), constrained_layout=True)
draw_network(axis, adjacency, industries, degrees, "Fixed Descriptive Industry Topology")
save_figure(figure, "mapping_public_topology")

,Item,Value
0,Public source,Kenneth R. French 49 Industry Portfolios
1,Stored archive,49_Industry_Portfolios_daily_CSV.zip
2,Archive bytes,"4,173,817"
3,SHA256,8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71...
4,Analysis start,2019-01-02
5,Analysis end,2023-12-29
6,Complete dates,"1,258"
7,Selected industries,12
8,Download during execution,None


,Industry,Mean daily return,Daily volatility,Degree
0,Agric,0.000728,0.019105,2
1,Food,0.000368,0.010999,3
2,Soda,0.000479,0.01305,2
3,Beer,0.000484,0.012889,4
4,Smoke,0.000462,0.014681,2
5,Toys,0.000312,0.023959,2
6,Fun,0.000512,0.020803,2
7,Books,0.000564,0.016741,3
8,Hshld,0.000445,0.012264,3
9,Clths,0.000537,0.019329,4


,First industry,Second industry,Correlation
0,Agric,Books,0.539326
1,Agric,Hlth,0.531205
2,Food,Beer,0.82577
3,Food,Smoke,0.654179
4,Food,Hshld,0.783311
5,Soda,Beer,0.821023
6,Soda,Hshld,0.755901
7,Beer,Smoke,0.638401
8,Beer,Hshld,0.824427
9,Toys,Fun,0.667338


,Quantity,Value
0,Industries,12
1,Undirected edges,16
2,Density,0.242424
3,Minimum degree,2
4,Mean degree,2.666667
5,Maximum degree,4


,Object,Role,Causal conclusion
0,Public returns,Fix correlations and the descriptive edge set,None
1,Graph edge,Defines which assignments enter a supplied exp...,None by itself
2,Generated assignment,Supplies known randomization probabilities,Design based within the generated experiment
3,Generated potential outcomes,Supply known exposure responses,Local to the declared design


<Figure size 840x700 with 2 Axes>

<Figure size 840x700 with 1 Axes>

## Mapping assignment vectors to neighbor exposure

Let $Z=(Z_1,\ldots,Z_N)$ be a randomized assignment vector, where $Z_j=1$ means that unit $j$ is assigned treatment. For a supplied adjacency matrix $A$, define unit $i$'s neighbor exposure as

$$
D_i(Z)=\mathbf 1\!\left\{\sum_{j=1}^{N}A_{ij}Z_j\geq1\right\}.
$$

The value $D_i(Z)=1$ means that at least one declared neighbor is assigned treatment. The mapping is a scientific choice: it states which part of the full assignment vector is used to define unit $i$'s potential outcome.

In [2]:
assignment_probability_value = 0.25
assignment_random_state = 7101
assignment_rng = np.random.default_rng(assignment_random_state)
assignment = assignment_rng.binomial(1, assignment_probability_value, len(industries))
realized_exposure = exposure_labels(adjacency, assignment)
probabilities = exposure_probabilities(adjacency, assignment_probability_value)

mapping_definition = pd.DataFrame([
    {"Symbol": "Z_j", "Definition": "Random assignment of unit j"},
    {"Symbol": "A_ij", "Definition": "Supplied undirected link between units i and j"},
    {"Symbol": "D_i(Z)", "Definition": "One when at least one declared neighbor of i is assigned"},
    {"Symbol": "pi_i(d)", "Definition": "Randomization probability that unit i has exposure d"},
])
save_table(mapping_definition, "mapping_definition", "Objects in the neighbor-exposure mapping.", "tab:cim7-mapping-definition", fit=True)

assignment_table = pd.DataFrame({
    "Industry": industries,
    "Degree": degrees,
    "Own assignment": assignment,
    "Assigned neighbors": adjacency @ assignment,
    "Realized exposure": realized_exposure,
})
save_table(assignment_table, "mapping_public_assignment", "One generated assignment on the fixed public topology.", "tab:cim7-mapping-public-assignment", fit=True)

probability_table = pd.DataFrame({
    "Industry": industries,
    "Degree": degrees,
    "Probability of no assigned neighbor": probabilities[:, 0],
    "Probability of at least one assigned neighbor": probabilities[:, 1],
    "Largest inverse probability": 1 / probabilities.min(axis=1),
})
save_table(probability_table, "mapping_public_probabilities", "Exact marginal exposure probabilities under independent assignment.", "tab:cim7-mapping-public-probabilities", fit=True)

assignment_contract = pd.DataFrame([
    {"Parameter": "Assignment probability", "Value": assignment_probability_value},
    {"Parameter": "Random state", "Value": assignment_random_state},
    {"Parameter": "Assigned industries", "Value": int(assignment.sum())},
    {"Parameter": "Industries with exposed neighbors", "Value": int(realized_exposure.sum())},
    {"Parameter": "Observed financial outcome used", "Value": "None"},
])
save_table(assignment_contract, "mapping_assignment_contract", "Declared generated assignment on the fixed public topology.", "tab:cim7-mapping-assignment-contract")

probability_plot = probability_table[["Industry", "Degree", "Probability of no assigned neighbor", "Probability of at least one assigned neighbor"]].copy()
probability_plot.to_csv(FIGURE_DATA / "mapping_public_probabilities.csv", index=False)
positions_bar = np.arange(len(industries))
figure, axis = plt.subplots(figsize=(9.0, 5.4), constrained_layout=True)
axis.bar(positions_bar, probabilities[:, 0], color=PALETTE["cyan"], label="No assigned neighbor")
axis.bar(positions_bar, probabilities[:, 1], bottom=probabilities[:, 0], color=PALETTE["amber"], label="At least one assigned neighbor")
axis.set_xticks(positions_bar, industries, rotation=50, ha="right")
axis.set_ylabel("Exposure probability")
axis.set_title("Exact Exposure Probabilities on the Fixed Topology")
axis.legend()
finish_axis(axis)
save_figure(figure, "mapping_public_probabilities")

assignment_plot = assignment_table.copy()
assignment_plot["X"] = positions[:, 0]
assignment_plot["Y"] = positions[:, 1]
assignment_plot.to_csv(FIGURE_DATA / "mapping_public_assignment_nodes.csv", index=False)
figure, axis = plt.subplots(figsize=(8.4, 7.0), constrained_layout=True)
draw_network(axis, adjacency, industries, realized_exposure, "Generated Neighbor Exposure on the Fixed Topology")
save_figure(figure, "mapping_public_assignment")

,Symbol,Definition
0,Z_j,Random assignment of unit j
1,A_ij,Supplied undirected link between units i and j
2,D_i(Z),One when at least one declared neighbor of i i...
3,pi_i(d),Randomization probability that unit i has expo...


,Industry,Degree,Own assignment,Assigned neighbors,Realized exposure
0,Agric,2,0,1,1
1,Food,3,0,0,0
2,Soda,2,0,0,0
3,Beer,4,0,0,0
4,Smoke,2,0,0,0
5,Toys,2,0,1,1
6,Fun,2,1,0,0
7,Books,3,0,1,1
8,Hshld,3,0,0,0
9,Clths,4,0,2,1


,Industry,Degree,Probability of no assigned neighbor,Probability of at least one assigned neighbor,Largest inverse probability
0,Agric,2,0.5625,0.4375,2.285714
1,Food,3,0.421875,0.578125,2.37037
2,Soda,2,0.5625,0.4375,2.285714
3,Beer,4,0.316406,0.683594,3.160494
4,Smoke,2,0.5625,0.4375,2.285714
5,Toys,2,0.5625,0.4375,2.285714
6,Fun,2,0.5625,0.4375,2.285714
7,Books,3,0.421875,0.578125,2.37037
8,Hshld,3,0.421875,0.578125,2.37037
9,Clths,4,0.316406,0.683594,3.160494


,Parameter,Value
0,Assignment probability,0.25
1,Random state,"7,101"
2,Assigned industries,3
3,Industries with exposed neighbors,6
4,Observed financial outcome used,None


<Figure size 900x540 with 1 Axes>

<Figure size 840x700 with 1 Axes>

## Exact Marginal Exposure Probabilities

Suppose assignments are independent with $\Pr(Z_j=1)=p$. If unit $i$ has degree $d_i$, then

$$
\pi_i(0)=(1-p)^{d_i},\qquad \pi_i(1)=1-(1-p)^{d_i}.
$$

Here, $\pi_i(0)$ is the probability that none of unit $i$'s neighbors is assigned, while $\pi_i(1)$ is the probability that at least one is assigned. Both probabilities must be positive for a contrast between the two exposure conditions.

In [3]:
ring = ring_graph(4)
ring_probability = 0.35
ring_assignments = list(itertools.product([0, 1], repeat=4))
ring_rows = []
for unit in range(4):
    exact_positive = sum(assignment_probability(vector, ring_probability) * exposure_labels(ring, vector)[unit] for vector in ring_assignments)
    formula_positive = exposure_probabilities(ring, ring_probability)[unit, 1]
    ring_rows.append({"Unit": unit, "Degree": int(ring[unit].sum()), "Enumerated probability": exact_positive, "Formula probability": formula_positive, "Absolute difference": abs(exact_positive - formula_positive)})
ring_probability_check = pd.DataFrame(ring_rows)
assert np.allclose(ring_probability_check["Enumerated probability"], ring_probability_check["Formula probability"])
save_table(ring_probability_check, "mapping_probability_check", "Enumeration check for marginal neighbor-exposure probabilities.", "tab:cim7-mapping-probability-check", fit=True)

section_summary = pd.DataFrame([
    {"Result": "Public topology", "Value": "12 industries and 16 descriptive edges"},
    {"Result": "Exposure rule", "Value": "At least one assigned neighbor"},
    {"Result": "Assignment source", "Value": "Generated independent Bernoulli draws"},
    {"Result": "Probability check", "Value": "Enumeration equals the closed form"},
    {"Result": "Causal network claim", "Value": "None from the public correlations"},
])
save_table(section_summary, "mapping_section_summary", "Summary of the fixed topology and exposure mapping.", "tab:cim7-mapping-section-summary", fit=True)

,Unit,Degree,Enumerated probability,Formula probability,Absolute difference
0,0,2,0.5775,0.5775,1.11022e-16
1,1,2,0.5775,0.5775,1.11022e-16
2,2,2,0.5775,0.5775,1.11022e-16
3,3,2,0.5775,0.5775,1.11022e-16


,Result,Value
0,Public topology,12 industries and 16 descriptive edges
1,Exposure rule,At least one assigned neighbor
2,Assignment source,Generated independent Bernoulli draws
3,Probability check,Enumeration equals the closed form
4,Causal network claim,None from the public correlations


,Result,Value
0,Public topology,12 industries and 16 descriptive edges
1,Exposure rule,At least one assigned neighbor
2,Assignment source,Generated independent Bernoulli draws
3,Probability check,Enumeration equals the closed form
4,Causal network claim,None from the public correlations
